# 02 - Preprocesado
## Clasificación de Marcha Patológica — Preparación de Datos

**Autor:** Weimar Andres Arenas Gonzalez  
**Curso:** Fundamentos de Deep Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WeimarArenas/ProyectoDL20261/blob/main/02%20-%20Preprocesado.ipynb)

---
**Objetivo:** Preparar los datos del dataset GIST para el entrenamiento de modelos de Deep Learning:
- Normalización de secuencias (StandardScaler)
- Selección de grupos de articulaciones (todos vs. solo piernas)
- Generación de splits LOSO-CV (Leave-One-Subject-Out)
- Guardado de datos listos para los modelos

In [ ]:
import subprocess, sys
pkgs = ['numpy', 'pandas', 'matplotlib', 'seaborn', 'scikit-learn']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=False)


## 1. Configuración del entorno

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from collections import Counter

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')

# Clases y articulaciones (mismas definiciones que notebook 01)
CLASS_NAMES = {0: 'Normal', 1: 'Antálgica', 2: 'Rígida', 3: 'Oscilante', 4: 'Steppage', 5: 'Trendelenburg'}

JOINT_NAMES = [
    'SpineBase', 'SpineMid', 'Neck', 'Head',
    'ShoulderLeft', 'ElbowLeft', 'WristLeft', 'HandLeft',
    'ShoulderRight', 'ElbowRight', 'WristRight', 'HandRight',
    'HipLeft', 'KneeLeft', 'AnkleLeft', 'FootLeft',
    'HipRight', 'KneeRight', 'AnkleRight', 'FootRight',
    'SpineShoulder', 'HandTipLeft', 'ThumbLeft', 'HandTipRight', 'ThumbRight'
]

# Grupos de articulaciones según el paper Jun et al. 2020
# "Legs only" → mejor resultado: 93.67% con GRU
LEG_JOINT_INDICES = [0, 1, 12, 13, 14, 15, 16, 17, 18, 19]  # Columna vertebral + piernas
ALL_JOINT_INDICES = list(range(25))  # Todas las articulaciones

print('Configuración lista.')

ModuleNotFoundError: No module named 'seaborn'

## 2. Carga de datos raw

Si el notebook 01 ya fue ejecutado, cargamos los .npy guardados. De lo contrario, re-ejecutamos la carga completa.

In [ ]:
# ============================================================
# FUNCIONES DE CARGA — Formato real del dataset GIST
# CSV: tab-separado, sin header, 101 columnas
# Col 0: timestamp | cols (1+j*4)+1,+2,+3 = x,y,z para joint j
# Directorio padre: human{N}_{gait_type}{rep}/
# ============================================================

import re as _re

# Indices de features: x,y,z de cada articulacion (omite timestamp y joint_id)
FEATURE_COLS = []
for _j in range(25):
    _base = 1 + _j * 4
    FEATURE_COLS.extend([_base + 1, _base + 2, _base + 3])
# Resultado: [2,3,4, 6,7,8, ..., 98,99,100]  -> 75 valores por fila

_GAIT_PAT = _re.compile(
    r'human(\d+)_(normal|antalgic|lurch|steppage|stiff_legged|trendelenburg)\d+'
)
CLASS_MAP = {
    'normal': 0, 'antalgic': 1, 'stiff_legged': 2,
    'lurch': 3, 'steppage': 4, 'trendelenburg': 5,
}
CLASS_NAMES = ['Normal', 'Antalgic', 'Stiff-legged', 'Lurching', 'Steppage', 'Trendelenburg']


def extract_label_and_subject(filepath):
    """Extrae clase y sujeto del nombre del directorio padre del CSV."""
    dirname = os.path.basename(os.path.dirname(filepath))
    m = _GAIT_PAT.match(dirname)
    if not m:
        return None, None
    return CLASS_MAP.get(m.group(2)), int(m.group(1))


def load_csv_sequence(filepath, skip_frames=10, n_frames=50):
    """
    Lee un CSV del GIST dataset (tab-separado, 101 cols, sin header).
    Retorna array (n_frames, 75) omitiendo los primeros skip_frames.
    """
    try:
        df = pd.read_csv(filepath, header=None, sep='\t')
        if df.shape[1] != 101 or len(df) < skip_frames + n_frames:
            return None
        seq = df.iloc[skip_frames:skip_frames + n_frames, FEATURE_COLS].values.astype(np.float32)
        return None if np.any(np.isnan(seq)) else seq
    except Exception:
        return None


def load_full_dataset(data_dir, verbose=True):
    """
    Carga todos los CSV validos del GIST dataset.
    data_dir: ruta a Pathological_Gaits/
    Retorna: X (N,50,75), y (N,), S (N,)
    """
    sequences, labels, subjects = [], [], []
    skipped = 0
    for dirname in sorted(os.listdir(data_dir)):
        dirpath = os.path.join(data_dir, dirname)
        if not os.path.isdir(dirpath):
            continue
        for fname in sorted(os.listdir(dirpath)):
            if not fname.endswith('.csv'):
                continue
            fpath = os.path.join(dirpath, fname)
            label, subject = extract_label_and_subject(fpath)
            if label is None:
                skipped += 1
                continue
            seq = load_csv_sequence(fpath)
            if seq is None:
                skipped += 1
                continue
            sequences.append(seq)
            labels.append(label)
            subjects.append(subject)
    if verbose:
        print(f'Secuencias cargadas: {len(sequences)} | Descartadas: {skipped}')
    return (np.array(sequences, dtype=np.float32),
            np.array(labels, dtype=np.int32),
            np.array(subjects, dtype=np.int32))


In [ ]:
# ---- Cargar datos (desde .npy de NB01 o directamente desde CSV) -------------
if not os.path.exists('pathological_gait_datasets'):
    os.system('git clone --quiet https://github.com/kooksung/pathological_gait_datasets.git')

DATA_DIR = 'pathological_gait_datasets/Pathological_Gaits'

if os.path.exists('X_raw.npy'):
    X = np.load('X_raw.npy')
    y = np.load('y_raw.npy')
    S = np.load('S_subjects.npy')
    print(f'Cargado desde X_raw.npy: X={X.shape}')
elif os.path.exists('X_all.npy'):
    X = np.load('X_all.npy')
    y = np.load('y_labels.npy')
    S = np.load('S_subjects.npy')
    print(f'Cargado desde X_all.npy: X={X.shape}')
else:
    assert os.path.isdir(DATA_DIR), f'No se encontro el dataset en: {DATA_DIR}'
    X, y, S = load_full_dataset(DATA_DIR)
    np.save('X_raw.npy', X)
    np.save('y_raw.npy', y)
    np.save('S_subjects.npy', S)
    print(f'Cargado desde CSV y guardado: X={X.shape}')

# Seleccion de joints de piernas
LEG_JOINT_INDICES = [0, 1, 12, 13, 14, 15, 16, 17, 18, 19]
leg_feat = [f for j in LEG_JOINT_INDICES for f in [j*3, j*3+1, j*3+2]]
X_legs = X[:, :, leg_feat]

print(f'X={X.shape} | X_legs={X_legs.shape}')
print(f'Clases: {dict(zip(*np.unique(y, return_counts=True)))}')
print(f'Sujetos: {sorted(np.unique(S))}')


## 3. Normalización de los datos

**Estrategia:** StandardScaler aplicado feature por feature (75 features).
El scaler se ajusta SOLO sobre datos de entrenamiento (dentro de cada fold LOSO-CV).

**Normalización global** (para inspección): centrar y escalar las 75 features.

In [ ]:
def normalize_sequences(X_train, X_test):
    """
    Normaliza los datos usando StandardScaler ajustado sobre X_train.
    
    Parámetros:
        X_train: (N_train, 50, 75)
        X_test:  (N_test, 50, 75)
    Retorna:
        X_train_norm, X_test_norm, scaler
    """
    N_train, T, F = X_train.shape
    N_test = X_test.shape[0]
    
    # Reshapear para el scaler: (N*T, F)
    X_train_2d = X_train.reshape(-1, F)
    X_test_2d = X_test.reshape(-1, F)
    
    scaler = StandardScaler()
    X_train_2d = scaler.fit_transform(X_train_2d)
    X_test_2d = scaler.transform(X_test_2d)
    
    return X_train_2d.reshape(N_train, T, F), X_test_2d.reshape(N_test, T, F), scaler


# Visualizar efecto de la normalización en un ejemplo
idx_example = 0
X_example = X[idx_example:100]
X_test_ex = X[100:120]

X_norm_ex, _, _ = normalize_sequences(X_example, X_test_ex)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(X_example[:, :, 36:39].flatten(), bins=60, color='steelblue', alpha=0.8)
axes[0].set_title('Antes de normalizar — HipLeft (x,y,z)', fontweight='bold')
axes[0].set_xlabel('Valor (metros)')

axes[1].hist(X_norm_ex[:, :, 36:39].flatten(), bins=60, color='darkorange', alpha=0.8)
axes[1].set_title('Después de normalizar (StandardScaler)', fontweight='bold')
axes[1].set_xlabel('Valor normalizado (σ)')

plt.tight_layout()
plt.savefig('efecto_normalizacion.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'Antes: media={X_example[:,:,36].mean():.3f}, std={X_example[:,:,36].std():.3f}')
print(f'Después: media={X_norm_ex[:,:,36].mean():.3f}, std={X_norm_ex[:,:,36].std():.3f}')

## 4. Selección de grupos de articulaciones

Según el paper, usar solo articulaciones de piernas mejora la precisión de 90.13% a 93.67%.
Se preparan dos versiones del dataset: con todas las articulaciones y solo con piernas.

In [ ]:
def select_joints(X, joint_indices):
    """
    Selecciona las articulaciones especificadas del tensor de secuencias.
    
    Parámetros:
        X: (N, 50, 75) — tensor completo
        joint_indices: lista de índices de joints a conservar
    Retorna:
        X_sel: (N, 50, len(joint_indices)*3)
    """
    # Expandir índices de joints a índices de features (x, y, z de cada joint)
    feature_indices = []
    for j in joint_indices:
        feature_indices.extend([j*3, j*3+1, j*3+2])
    return X[:, :, feature_indices]


# Crear versiones del dataset
X_all = X.copy()  # Todos los joints: (N, 50, 75)
X_legs = select_joints(X, LEG_JOINT_INDICES)  # Solo piernas: (N, 50, 30)

print(f'X con todos los joints:  {X_all.shape}  → {X_all.shape[2]} features/frame')
print(f'X con solo piernas:      {X_legs.shape}  → {X_legs.shape[2]} features/frame')

print(f'\nArticulaciones incluidas en grupo piernas:')
for j in LEG_JOINT_INDICES:
    print(f'  Joint {j:2d}: {JOINT_NAMES[j]}')

## 5. Generación de splits LOSO-CV

**Leave-One-Subject-Out Cross-Validation (LOSO-CV):** Consistente con Jun et al. (2020).
En cada fold, los datos de UN sujeto se usan para test y el resto para train.
Con 10 sujetos → 10 folds.

In [ ]:
def generate_loso_splits(X, y, S):
    """
    Genera los 10 folds de LOSO-CV.
    
    Retorna lista de diccionarios con:
        {'fold': int, 'test_subject': int,
         'X_train': array, 'y_train': array,
         'X_test': array,  'y_test': array}
    """
    logo = LeaveOneGroupOut()
    splits = []
    
    for fold_idx, (train_idx, test_idx) in enumerate(logo.split(X, y, groups=S)):
        test_subject = np.unique(S[test_idx])[0]
        
        X_train_raw = X[train_idx]
        X_test_raw = X[test_idx]
        y_train = y[train_idx]
        y_test = y[test_idx]
        
        # Normalizar (scaler ajustado solo sobre train del fold)
        X_train_norm, X_test_norm, scaler = normalize_sequences(X_train_raw, X_test_raw)
        
        splits.append({
            'fold': fold_idx,
            'test_subject': int(test_subject),
            'train_idx': train_idx,
            'test_idx': test_idx,
            'X_train': X_train_norm,
            'X_test': X_test_norm,
            'y_train': y_train,
            'y_test': y_test,
            'scaler': scaler
        })
    
    return splits


print('Generando splits LOSO-CV para todos los joints...')
loso_splits_all = generate_loso_splits(X_all, y, S)

print('Generando splits LOSO-CV para joints de piernas...')
loso_splits_legs = generate_loso_splits(X_legs, y, S)

print(f'\nTotal de folds: {len(loso_splits_all)}')
print('\nResumen por fold:')
print(f'{"Fold":>5} {"Sujeto Test":>12} {"Train":>8} {"Test":>6} {"Clases Test":>15}')
print('-' * 55)
for sp in loso_splits_all:
    cls_dist = Counter(sp['y_test'])
    print(f"{sp['fold']:>5} {sp['test_subject']:>12} {len(sp['y_train']):>8} {len(sp['y_test']):>6}  {dict(cls_dist)}")

In [ ]:
# Visualizar la distribución de clases en un fold típico
fold_example = loso_splits_all[0]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, split_key, title in zip(axes, ['y_train', 'y_test'],
                                 [f'Train (sujetos 2-10)', f'Test (sujeto {fold_example["test_subject"]})']):
    counts = Counter(fold_example[split_key])
    labels = [CLASS_NAMES[i] for i in sorted(counts.keys())]
    values = [counts[i] for i in sorted(counts.keys())]
    bars = ax.bar(labels, values, color=plt.cm.Set2(np.linspace(0, 1, 6)), edgecolor='black')
    ax.set_title(f'Fold 0 — {title}', fontweight='bold')
    ax.set_xlabel('Clase')
    ax.set_ylabel('Muestras')
    ax.tick_params(axis='x', rotation=30)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, str(val),
                ha='center', va='bottom', fontsize=9)

plt.suptitle('Distribución de clases en Fold 0 (LOSO-CV)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('loso_cv_splits.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. Análisis de correlación entre features

Se analiza la correlación entre features para entender la redundancia y confirmar que la selección de joints de piernas es informativa.

In [ ]:
# Calcular correlación entre las 75 features (usando media temporal)
# Reshape: (N*T, F) para calcular correlación entre features
X_2d = X.reshape(-1, 75)

# Calcular correlación entre features de joints de piernas vs. brazos
# Features de piernas: joints 0,1,12-19 → columnas [0:3, 3:6, 36:60] aprox
leg_feature_cols = []
for j in LEG_JOINT_INDICES:
    leg_feature_cols.extend([j*3, j*3+1, j*3+2])

X_legs_2d = X_2d[:, leg_feature_cols]

# Correlación entre las features de piernas
corr_matrix = np.corrcoef(X_legs_2d.T)

leg_feature_names = []
for j in LEG_JOINT_INDICES:
    jname = JOINT_NAMES[j]
    leg_feature_names.extend([f'{jname[:6]}_x', f'{jname[:6]}_y', f'{jname[:6]}_z'])

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=False, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            xticklabels=leg_feature_names, yticklabels=leg_feature_names,
            ax=ax, linewidths=0.1)
ax.set_title('Matriz de correlación — Features de articulaciones de piernas',
             fontsize=12, fontweight='bold')
plt.xticks(rotation=75, ha='right', fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig('correlacion_features_piernas.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. One-Hot Encoding de etiquetas

In [ ]:
from tensorflow.keras.utils import to_categorical

N_CLASSES = 6

# Verificar que las etiquetas están en [0, N_CLASSES-1]
print(f'Rango de etiquetas: [{y.min()}, {y.max()}]')
print(f'Clases únicas: {np.unique(y)}')

# Ejemplo de one-hot encoding
y_onehot_example = to_categorical(y[:5], num_classes=N_CLASSES)
print(f'\nEjemplo: y[:5] = {y[:5]}')
print(f'One-hot:') 
print(y_onehot_example)
print(f'\n(One-hot se aplica dentro de cada fold durante el entrenamiento)')

## 8. Guardado de datos preprocesados

In [ ]:
# Guardar tensores preprocesados y metadata
# Los splits LOSO-CV se regeneran en cada notebook de modelo
# (el scaler se ajusta fresh en cada fold)

np.save('X_all.npy', X_all)
np.save('X_legs.npy', X_legs)
np.save('y_labels.npy', y)
np.save('S_subjects.npy', S)

# Guardar metadata del preprocesado
metadata = {
    'n_samples': len(X),
    'n_frames': X.shape[1],
    'n_features_all': X_all.shape[2],
    'n_features_legs': X_legs.shape[2],
    'n_classes': N_CLASSES,
    'n_subjects': len(np.unique(S)),
    'class_names': CLASS_NAMES,
    'joint_names': JOINT_NAMES,
    'leg_joint_indices': LEG_JOINT_INDICES,
    'skip_frames': 10,
}

with open('metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

print('=== DATOS GUARDADOS ===')
print(f'  X_all.npy:      {X_all.shape}  ({X_all.nbytes/1e6:.1f} MB)')
print(f'  X_legs.npy:     {X_legs.shape}  ({X_legs.nbytes/1e6:.1f} MB)')
print(f'  y_labels.npy:   {y.shape}')
print(f'  S_subjects.npy: {S.shape}')
print(f'  metadata.pkl:   guardado')

print('\n=== RESUMEN DEL PREPROCESADO ===')
print(f'  Dataset: {len(X)} secuencias × 50 frames × 75 features')
print(f'  Normalización: StandardScaler ajustado por fold (NO global)')
print(f'  Estrategia CV: LOSO-CV con {len(np.unique(S))} sujetos')
print(f'  Dos variantes: todos los joints (75) y solo piernas (30)')
print(f'\n→ Siguiente: Notebook 03 - Arquitectura línea de base RNN')

In [ ]:
# Verificación final: dimensiones de un fold
sp = loso_splits_all[0]
print(f'Fold 0 (sujeto test = {sp["test_subject"]}):')
print(f'  X_train: {sp["X_train"].shape}')
print(f'  X_test:  {sp["X_test"].shape}')
print(f'  y_train: {sp["y_train"].shape}, distribución: {Counter(sp["y_train"])}')
print(f'  y_test:  {sp["y_test"].shape},  distribución: {Counter(sp["y_test"])}')
print(f'\n  Datos correctamente normalizados:')
print(f'  Train — media: {sp["X_train"].mean():.4f}, std: {sp["X_train"].std():.4f}')
print(f'  Test  — media: {sp["X_test"].mean():.4f},  std: {sp["X_test"].std():.4f}')